# MLOps Demo - Data Exploration

This notebook provides an interactive exploration of the wine dataset used in our MLOps demo.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Set style
plt.style.use('default')
sns.set_palette("husl")

print("Libraries imported successfully!")

In [ ]:
# Load the wine dataset
wine_data = load_wine()
X, y = wine_data.data, wine_data.target

# Create DataFrame for easier manipulation
df = pd.DataFrame(X, columns=wine_data.feature_names)
df['target'] = y
df['target_name'] = [wine_data.target_names[i] for i in y]

print(f"Dataset shape: {df.shape}")
print(f"Features: {len(wine_data.feature_names)}")
print(f"Classes: {wine_data.target_names}")

df.head()

In [ ]:
# Dataset overview
print("=== Dataset Information ===")
print(f"Total samples: {len(df)}")
print(f"Features: {len(wine_data.feature_names)}")
print(f"Classes: {len(wine_data.target_names)}")
print(f"Missing values: {df.isnull().sum().sum()}")

print("\n=== Class Distribution ===")
class_counts = df['target_name'].value_counts()
print(class_counts)

# Visualize class distribution
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='target_name')
plt.title('Wine Class Distribution')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Feature statistics
print("=== Feature Statistics ===")
feature_stats = df[wine_data.feature_names].describe()
print(feature_stats)

# Check feature scales
feature_ranges = feature_stats.loc['max'] - feature_stats.loc['min']
print(f"\nFeature scale variation: {feature_ranges.max() / feature_ranges.min():.2f}x")
print("Note: Large scale differences suggest feature scaling might be beneficial")

In [ ]:
# Feature correlation heatmap
plt.figure(figsize=(12, 10))
correlation_matrix = df[wine_data.feature_names].corr()
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

# Find highly correlated features
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.7:
            high_corr_pairs.append((
                correlation_matrix.columns[i],
                correlation_matrix.columns[j],
                correlation_matrix.iloc[i, j]
            ))

print("\n=== Highly Correlated Feature Pairs (|r| > 0.7) ===")
for feat1, feat2, corr in high_corr_pairs:
    print(f"{feat1} <-> {feat2}: {corr:.3f}")

In [ ]:
# Feature distributions by class
# Select a few key features for visualization
key_features = ['alcohol', 'flavanoids', 'color_intensity', 'proline']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, feature in enumerate(key_features):
    sns.boxplot(data=df, x='target_name', y=feature, ax=axes[i])
    axes[i].set_title(f'{feature} by Wine Class')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Quick model training and evaluation
print("=== Quick Model Training ===")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# Train a simple Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions
y_pred = rf_model.predict(X_test)
accuracy = rf_model.score(X_test, y_test)

print(f"\nModel Accuracy: {accuracy:.4f}")

# Classification report
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=wine_data.target_names))

In [ ]:
# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=wine_data.target_names,
            yticklabels=wine_data.target_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': wine_data.feature_names,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=feature_importance.head(10), x='importance', y='feature')
plt.title('Top 10 Most Important Features')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.show()

print("=== Top 5 Most Important Features ===")
for i, row in feature_importance.head().iterrows():
    print(f"{row['feature']}: {row['importance']:.4f}")

In [ ]:
# Sample prediction for API testing
print("=== Sample Prediction ===")
sample_idx = 0
sample_features = X_test[sample_idx]
sample_prediction = rf_model.predict([sample_features])[0]
sample_probabilities = rf_model.predict_proba([sample_features])[0]

print(f"Sample features: {sample_features.tolist()}")
print(f"True label: {wine_data.target_names[y_test[sample_idx]]}")
print(f"Predicted label: {wine_data.target_names[sample_prediction]}")
print(f"Prediction probabilities: {sample_probabilities}")

print("\n=== API Testing Format ===")
api_payload = {
    "features": sample_features.tolist()
}
print(f"API payload: {api_payload}")
print("\nYou can use this payload to test the API endpoints!")

## Summary

This exploration shows that the wine dataset is well-suited for our MLOps demo:

1. **Clean Data**: No missing values, consistent format
2. **Balanced Classes**: Reasonably balanced class distribution
3. **Good Separability**: High model accuracy (>90%)
4. **Rich Features**: 13 chemical features with clear importance patterns
5. **Real-world Complexity**: Feature correlations and scaling challenges

The model performance suggests that our MLOps pipeline will have meaningful predictions to monitor and track.

## Next Steps

1. Run the main MLOps pipeline: `modal run main.py pipeline`
2. Start the API server: `modal serve main.py::serve_model`
3. Test with the sample payload above
4. Monitor predictions and drift detection